# README

- **Author**: RUEI-HSIANG, CHANG（B12303030；AI協助）
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?

第七輪：將9位候選人的2022公報個人欄位，經當屆職務、學校及參選情境交叉支持後，接回生日、明載性別、學經歷及非首次欄位。不是單以姓名合併。保留陳若翠跨選區、湯茹婷政黨變動；名冊黨派不改。另保存9個政黨標示人物頁入口：只有3頁已有實質學經歷，其他6頁仍待更新，不算已完成背景頁。黃裕庭頁面明載男性，自述性別分來源保存，不從照片或姓名推測。

## What does this file take?

下方PINS固定4個前輪快照、13人公報分欄、逐人核對計畫及3份來源manifest。每份具名原始來源依manifest／公報表所存hash再核對；原始來源文件：`data/raw/identity_round7_20261019_documentation.ipynb`。核對計畫為本地人工審閱設定，不冒充原始來源。

## What does this file output?

- 3份current：master1583人、field31660列、council1502人。
- `council_history_pending_identity_current.csv`：從162列移除本輪9個已支持身分連結，餘153列。
- `council_identity_review_round7_20261019.csv`：原11個公報待核對者逐人判斷，9支持、2仍待查。
- `party_profile_review_round7_20261019.csv`：9頁來源狀態，3頁實質背景與6頁占位內容分開。
- `candidate_evidence_round7_20261019.csv`：逐候選人×欄位×來源觀察，不是完整分析樣本。

輸出角色為review_inventory／source_observation。整理後文件：`data/processed/identity_round7_20261019_documentation.ipynb`。筆數、schema、主鍵與磁碟輸出hash在最後一格讀回驗證。社群貼文、窗口計數與追蹤數仍暫緩。


第八輪接續：本輪4份current輸出已封存至pre_cross_office_round8快照，避免舊程式覆寫最新結果；7份本輪輸出仍可獨立驗證。

In [1]:
from pathlib import Path
import pandas as pd,json,hashlib,re,unicodedata,csv,datetime
from bs4 import BeautifulSoup
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def read(p):return pd.read_csv(ROOT/p,dtype=str,keep_default_na=False)
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',str(s))).replace('台','臺').replace('温','溫')
PINS={'data/processed/snapshots/pre_identity_round7_20261019/candidate_master_current.csv': '863118696723c57e4188dea85a3a344658641e3f12692a8c87862e7a1ee1bd12', 'data/processed/snapshots/pre_identity_round7_20261019/candidate_fields_current.csv': '7344f7328301b184d7f2c89473c0c0532e1d07e9e3f64a739df6f3c528c60350', 'data/processed/snapshots/pre_identity_round7_20261019/council_background_20261019.csv': '5313bcddabc8f33f824be67bf86d04d78b0000a24541133f0e0857d9a1319513', 'data/processed/snapshots/pre_identity_round7_20261019/council_history_pending_identity_current.csv': 'ca2335dd884a62a8d9b6689cb25c7ca1cea32707911e43ce5ef7deffe2cbbe00', 'data/processed/council_bulletin_cells_round5_20261019.csv': 'c62a180244de8991791688276dd40552fb054708bb36a838fd5df6fe9315b35c', 'data/code/identity_followup_review_20261019.json': 'fc0f4dab6e3cf653fb65204b79a9bcf5753cb2bc4eb4a175357bba4b6a99dcc2', 'data/raw/identity_followup_20261019/manifest.json': 'fab5cd9be7cdd78eeb3f8149dfdb6f2de60c000679cafd2b052b158de343853d', 'data/raw/identity_followup_20261019/web_manifest.json': '834b0328d6458eaf7c1138e5466db42771c57e5a0b9219590581f7d300c7ea3f', 'data/raw/kmtks_web_20261019/manifest.json': 'a519c5d80a86591bec6da77b7f031e56c5c10fa9e79245e5e1b74332a4709fb8'}
for p,h in PINS.items():assert (ROOT/p).is_file() and sha(ROOT/p)==h,p
S='data/processed/snapshots/pre_identity_round7_20261019'
master=read(S+'/candidate_master_current.csv');fields=read(S+'/candidate_fields_current.csv');council=read(S+'/council_background_20261019.csv');pending=read(S+'/council_history_pending_identity_current.csv');cells=read('data/processed/council_bulletin_cells_round5_20261019.csv');plan=json.loads((ROOT/'data/code/identity_followup_review_20261019.json').read_text())
assert len(master)==1583 and len(fields)==31660 and len(council)==1502 and len(pending)==162 and len(cells)==13 and len(plan)==9
captures=[]
for p in ['data/raw/identity_followup_20261019/manifest.json','data/raw/identity_followup_20261019/web_manifest.json','data/raw/kmtks_web_20261019/manifest.json']:captures+=json.loads((ROOT/p).read_text())
for a in captures:
 if a.get('path'):assert (ROOT/a['path']).is_file() and sha(ROOT/a['path'])==a['sha256'],a['path']
for p,h in cells[['source_file','source_sha256']].drop_duplicates().itertuples(index=False):assert sha(ROOT/p)==h,p
for p,h in pending[['history_file','source_sha256']].drop_duplicates().itertuples(index=False):assert sha(ROOT/p)==h,p

def source_text(p):
 s=(ROOT/p).read_text(errors='replace')
 if p.endswith('.html'):return BeautifulSoup(s,'html.parser').get_text(' ',strip=True)
 s=re.sub(r'cite[^†]*†([^]*)',r'\1',s)
 return re.sub(r'L\d+:\s*','',s)
def rowkey(k):
 d=master[master.candidate_key.eq(k)];assert len(d)==1,k
 return d.iloc[0]
def setfield(key,f,value,status,src,note):
 mask=fields.candidate_key.eq(key)&fields.field.eq(f);assert mask.sum()==1
 fields.loc[mask,['existing_value','status','evidence','note']]=[value,status,src,note]
 master.loc[master.candidate_key.eq(key),f+'_status']=status
obs=[]
def evidence(key,f,value,url,path,status,year):
 p=rowkey(key);obs.append({'candidate_key':key,'candidate_name':p.candidate_name,'field':f,'value':value,'source_url':url,'source_file':path,'source_year':str(year),'evidence_status':status,'checked_date':'2026-10-19'})
print('Fixed inputs and source bytes checked; no downloads run by this notebook.')

Fixed inputs and source bytes checked; no downloads run by this notebook.


In [2]:

review=[];accepted=[];new_nonfirst_any=[];new_nonfirst_council=[]
council['round7_identity_url']='';council['party_page_lead_url']=''
for a in plan:
 name=a['candidate_name'];d=cells[cells.candidate_name.eq(name)];assert len(d)==1
 c=d.iloc[0];key=c.candidate_key;p=rowkey(key)
 assert p.county_city=='高雄市' and p.office_type=='直轄市議員'
 body=source_text(a['source_file']);assert norm(name) in norm(body) and sha(ROOT/a['source_file'])==a['source_sha256']
 for claim in a['identity_anchor_claims']:assert norm(claim) in norm(body),(name,claim)
 h=pending[pending.candidate_key.eq(key)];assert len(h)==1
 h=h.iloc[0];hist=list(csv.reader((ROOT/h.history_file).open(encoding='utf-8-sig')))[int(h.history_row)-1];hist=[v.strip().lstrip("'") for v in hist]
 assert hist[6]==name and hist[8] in ['1','2']
 roc=hist[9].zfill(7);birth=datetime.date(int(roc[:3])+1911,int(roc[3:5]),int(roc[5:])).isoformat();sex={'1':'男','2':'女'}[hist[8]]
 assert birth==c.birth_date_declared==h.birth_date_history_candidate and sex==c.sex_raw==h.history_sex
 # Source page/row and independent 2026 context are both retained; not merely name matching.
 mask=council.candidate_key.eq(key);old=council[mask].iloc[0]
 assert old.birth_date_supported_history=='' and old.sex_supported_history==''
 for col,value in {'birth_date_supported_history':birth,'birth_year_supported_history':birth[:4],'sex_supported_history':sex,'birth_sex_evidence_status':'2022官方公報與當屆履歷／參選情境交叉支持；非2026戶籍重新認定','history_source_file':h.history_file,'history_source_row':h.history_row,'history_identity_review':a['review']+'；'+a['cross_district_note'],'round7_identity_url':a['source_url'],'bulletin_identity_status':'2022_person_cells_and_2026_biographical_context_supported'}.items():council.loc[mask,col]=value
 if not old.education_raw:council.loc[mask,'education_raw']=c.education_raw
 if not old.experience_raw:council.loc[mask,'experience_raw']=c.experience_raw
 if not old.background_source_url:council.loc[mask,'background_source_url']=c.source_url
 note=a['review']+'；'+a['cross_district_note']+'；當屆佐證來源方法：'+a['source_retrieval_method']
 for f,value in [('birth_date',birth),('sex',sex),('education',c.education_raw),('experience',c.experience_raw)]:
  evidence(key,f,value,c.source_url,c.source_file,'2022官方公報個人欄位；'+note,2022)
  if f!='experience':setfield(key,f,value,'2022官方公報個人欄位與當屆履歷有身分連結支持',c.source_url+' | '+a['source_url'],note+'；學歷為歷史申報原文，不認證學位')
 evidence(key,'identity_link',' | '.join(a['identity_anchor_claims']),a['source_url'],a['source_file'],note,2026)
 for f,col,out in [('first_time_any_office','first_time_any_office_supported',new_nonfirst_any),('first_time_this_office','first_time_this_office_supported',new_nonfirst_council)]:
  if old[col]=='':out.append(name)
  council.loc[mask,col]='0';setfield(key,f,'0','有已連結2022議員參選紀錄，支持非終生首次',c.source_url+' | '+a['source_url'],note+'；不以是否當選判定')
  evidence(key,f,'0',c.source_url,c.source_file,'2022議員參選及身分支持，不推定2026仍在任',2022)
 setfield(key,'profile',c.source_url,'已連結2022公報個人欄位；網頁人物頁可得性另列',c.source_url+' | '+a['source_url'],note+'；不把新聞中的其他人履歷套入本人')
 accepted.append(key)
 review.append({'candidate_key':key,'candidate_name':name,'decision':'supported_2022_person_cells_link','birth_date':birth,'sex':sex,'official_source_url':c.source_url,'official_source_file':c.source_file,'official_source_page':c.page,'history_source_file':h.history_file,'history_source_row':h.history_row,'identity_source_url':a['source_url'],'identity_source_file':a['source_file'],'identity_source_method':a['source_retrieval_method'],'identity_anchors':' | '.join(a['identity_anchor_claims']),'cross_district_note':a['cross_district_note'],'checked_date':'2026-10-19'})
for name in ['李柏融','柳淑芳']:
 c=cells[cells.candidate_name.eq(name)].iloc[0]
 review.append({'candidate_key':c.candidate_key,'candidate_name':name,'decision':'still_pending_current_identity_corroboration','birth_date':'','sex':'','official_source_url':c.source_url,'official_source_file':c.source_file,'official_source_page':c.page,'history_source_file':'','history_source_row':'','identity_source_url':'','identity_source_file':'','identity_source_method':'requests404_and_tool_open_failed; no false completion','identity_anchors':'','cross_district_note':'','checked_date':'2026-10-19'})
assert len(accepted)==9
pending=pending[~pending.candidate_key.isin(accepted)].copy();assert len(pending)==153
review=pd.DataFrame(review);assert len(review)==11 and review.candidate_key.is_unique
print('New official-history person links:',len(accepted),'remaining older identity worklist:',len(pending))

New official-history person links: 9 remaining older identity worklist: 153


In [3]:

# Nine candidate page entries; six lack substantive personal background and are not counted complete.
party=[];new_profile_keys=[]
manifest=json.loads((ROOT/'data/raw/kmtks_web_20261019/manifest.json').read_text())
cn={1:'一',2:'二',3:'三',4:'四',5:'五',6:'六',7:'七',8:'八',9:'九',10:'十',11:'十一',12:'十二',13:'十三',14:'十四',15:'十五'}
for a in manifest:
 raw=(ROOT/a['path']).read_text();name=a['candidate_name'];hits=council[council.candidate_name.eq(name)&council.county_city.eq('高雄市')];assert len(hits)==1
 c=hits.iloc[0];key=c.candidate_key;district=int(re.search(r'第(\d+)選舉區',c.electoral_district).group(1));assert '第'+cn[district]+'選區' in raw and '# '+name in raw
 # Extract only explicitly headed education / experience. Generic campaign promises are not career evidence.
 lines=[re.sub(r'^L\d+:\s*','',z) for z in raw.splitlines()]
 sections={'education_raw':[],'experience_raw':[]};current=None
 for line in lines:
  if line.strip()=='### 學歷':current='education_raw';continue
  if line.strip()=='### 經歷':current='experience_raw';continue
  if line.startswith('### ') or line.startswith('## ') or '資料由各辦公室自行維護' in line:current=None
  if current and line.lstrip().startswith('* '):sections[current].append(line.lstrip()[2:].strip())
 edu=' | '.join(sections['education_raw']);exp=' | '.join(sections['experience_raw']);substantive=bool(edu or exp)
 assert substantive==bool(a['has_substantive_background'])
 mask=council.candidate_key.eq(key);council.loc[mask,'party_page_lead_url']=a['url']
 setfield(key,'website',a['url'],'政黨標示候選人頁入口，是否有實質履歷另列',a['url'],'web工具文字擷取，非原始HTTP下載；不把共用頁尾當個人帳號')
 if substantive:
  old=council[mask].iloc[0]
  if not any(old[x] for x in ['moi_profile_url','campaign_profile_url','additional_profile_url']):
   new_profile_keys.append(key);council.loc[mask,['additional_profile_url','additional_profile_status']]=[a['url'],'2026 party-labeled candidate page with substantive self-reported background; not degree certification']
  for f,value in [('education',edu),('experience',exp)]:
   if not value:continue
   evidence(key,f,value,a['url'],a['path'],'2026政黨標示候選人頁，由辦公室自行維護；非獨立學位或任職認證',2026)
   col=f+'_raw'
   if not old[col]:
    council.loc[mask,col]=value
    if not old.background_source_url:council.loc[mask,'background_source_url']=a['url']
    if f=='education':setfield(key,f,value,'候選人頁自述原文已有，學位尚未獨立核實',a['url'],'學校清單不等於畢業；保留原文與來源層級')
  setfield(key,'profile',a['url'],'已有實質候選人背景頁，來源為自述',a['url'],'學經歷未獨立認證；原2022官方資料另表保留')
  if name=='黃裕庭':
   assert '男性候選人' in raw and '第3、4 屆仁政里里長' in raw
   if old.sex_supported_history=='' and old.sex_additional_source_value=='':
    council.loc[mask,['sex_additional_source_value','sex_additional_source_url','sex_additional_source_year']]=['男',a['url'],'2026']
    setfield(key,'sex','男','候選人頁明載男性，屬自述來源',a['url'],'明載男性候選人；未由姓名、照片或職務推測；非戶籍認證')
    evidence(key,'sex','男',a['url'],a['path'],'candidate_explicit_male_self_description_not_government',2026)
   if old.first_time_any_office_supported=='':
    council.loc[mask,'first_time_any_office_supported']='0';new_nonfirst_any.append(name)
    setfield(key,'first_time_any_office','0','候選人頁明載過往當選里長職務支持非首次',a['url'],'第3、4屆仁政里里長；不把里長經歷直接當作曾參選議員')
    evidence(key,'first_time_any_office','0',a['url'],a['path'],'explicit_prior_elected_village_head_role_self_report',2026)
 party.append({'candidate_key':key,'candidate_name':name,'electoral_district':c.electoral_district,'source_url':a['url'],'source_file':a['path'],'source_method':a['retrieval_method'],'education_raw':edu,'experience_raw':exp,'has_substantive_background':str(substantive),'is_placeholder_only':str(not substantive),'decision':'accepted_person_background_source_self_reported' if substantive else 'entry_only_background_placeholder_not_completed','checked_date':'2026-10-19'})
party=pd.DataFrame(party);assert len(party)==9 and party.has_substantive_background.eq('True').sum()==3 and len(new_profile_keys)==3
assert len(master)==1583 and len(fields)==31660 and len(council)==1502
print('Candidate pages read:',len(party),'substantive:',len(new_profile_keys),'placeholder-only:',int(party.is_placeholder_only.eq('True').sum()))

Candidate pages read: 9 substantive: 3 placeholder-only: 6


In [4]:

evidence_table=pd.DataFrame(obs)
outputs={
 'data/processed/snapshots/pre_cross_office_round8_20261019/candidate_master_current.csv':master,
 'data/processed/snapshots/pre_cross_office_round8_20261019/candidate_fields_current.csv':fields,
 'data/processed/snapshots/pre_cross_office_round8_20261019/council_background_20261019.csv':council,
 'data/processed/snapshots/pre_cross_office_round8_20261019/council_history_pending_identity_current.csv':pending,
 'data/processed/council_identity_review_round7_20261019.csv':review,
 'data/processed/party_profile_review_round7_20261019.csv':party,
 'data/processed/candidate_evidence_round7_20261019.csv':evidence_table}
checks={}
for path,d in outputs.items():
 target=(ROOT/path).resolve();assert target.is_relative_to((ROOT/'data/processed').resolve()) and path not in PINS
 # Preserve explicit source line breaks as literal sequences so CSV records stay on one physical line.
 d=(d.map if hasattr(d,'map') else d.applymap)(lambda z:str(z).replace('\r','').replace('\n','\\n'))
 d.to_csv(target,index=False,encoding='utf-8-sig');saved=read(path);assert list(saved.columns)==list(d.columns) and saved.shape==d.shape
 assert len(target.read_text(encoding='utf-8-sig').splitlines())==len(saved)+1,path
 key=['candidate_key','field'] if path.endswith('candidate_fields_current.csv') else ['candidate_key'] if path not in ['data/processed/candidate_evidence_round7_20261019.csv'] else []
 if key:assert not saved.duplicated(key).any(),path
 checks[path]={'rows':len(saved),'columns':len(saved.columns),'key':key,'sha256':sha(target)}
assert int(council.birth_date_supported_history.ne('').sum())==701
assert int(council.sex_supported_history.ne('').sum())==701
assert int((council.moi_profile_url.eq('')&council.campaign_profile_url.eq('')&council.additional_profile_url.eq('')).sum())==730
before=read(S+'/council_background_20261019.csv');assert council[['candidate_key','party_registration_raw','incumbency_2026']].equals(before[['candidate_key','party_registration_raw','incumbency_2026']])
for name in ['李柏融','柳淑芳']:
 c=council[council.candidate_name.eq(name)].iloc[0];assert c.birth_date_supported_history=='' and c.sex_supported_history==''
summary={'new_history_people':9,'new_history_names':review[review.decision.eq('supported_2022_person_cells_link')].candidate_name.tolist(),'historical_2022_birth_dates':701,'historical_2022_sex':701,'birth_official_union':int((council.birth_date_supported_history.ne('')|council.birth_date_additional_official.ne('')).sum()),'sex_explicit_source_union':int((council.sex_supported_history.ne('')|council.sex_additional_source_value.ne('')).sum()),'new_substantive_profile_people':3,'remaining_web_profile_gaps':730,'profile_pages_placeholder_only':6,'council_nonfirst_any':int(council.first_time_any_office_supported.eq('0').sum()),'council_nonfirst_council':int(council.first_time_this_office_supported.eq('0').sum()),'new_nonfirst_any_names':new_nonfirst_any,'remaining_identity_worklist':len(pending),'outputs':checks}
(ROOT/'data/code/council_identity_round7_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2));print(json.dumps(summary,ensure_ascii=False,indent=2))

{
  "new_history_people": 9,
  "new_history_names": [
    "林慧欣",
    "鄭東元",
    "陳若翠",
    "陳惠君",
    "童燕珍",
    "蘇致榮",
    "徐尚賢",
    "湯茹婷",
    "洪村銘"
  ],
  "historical_2022_birth_dates": 701,
  "historical_2022_sex": 701,
  "birth_official_union": 705,
  "sex_explicit_source_union": 706,
  "new_substantive_profile_people": 3,
  "remaining_web_profile_gaps": 730,
  "profile_pages_placeholder_only": 6,
  "council_nonfirst_any": 712,
  "council_nonfirst_council": 704,
  "new_nonfirst_any_names": [
    "林慧欣",
    "鄭東元",
    "陳若翠",
    "陳惠君",
    "童燕珍",
    "蘇致榮",
    "徐尚賢",
    "湯茹婷",
    "洪村銘",
    "黃裕庭"
  ],
  "remaining_identity_worklist": 153,
  "outputs": {
    "data/processed/snapshots/pre_cross_office_round8_20261019/candidate_master_current.csv": {
      "rows": 1583,
      "columns": 34,
      "key": [
        "candidate_key"
      ],
      "sha256": "0d2d2620f62cde3abaf691ce843925231bc84f95362562b16794d74aadf022e9"
    },
    "data/processed/snapshots/pre_cross_office_round8_2